### HOW TO USE THIS NOTEBOOK
This document is the master template, so it is in **"View Only"** mode and you cannot run the code directly on this link.

To run your own analyses and execute the code, follow these steps:
1. Click **File > Save a copy in Drive** from the top-left menu.
2. The copy that opens in a new tab will be **entirely your own**. You can safely edit, change and run all the code from that file.

**Tip:** If you get confused with parameters while working in your own copy, or want to double-check the original code structure, keep this first tab open. It can stay open as a handy **"Reference Guide"** throughout the analysis.


---

# Welcome to the World of Applied Bioinformatics: Genomic Data Analyses (BIF201)
## DAY 2, LESSON 4: Setting Up the Raw Data Quality Control (QC) Workflow

### OUR GOAL FOR THIS LESSON:
Starting from a real, current scientific publication, tracking down raw data in public databases (NCBI SRA/ENA) using accession numbers. Pulling the data directly into the Colab cloud environment without downloading it to our own computer. Producing our first quality report by taking an "X-ray" of our short-read (Illumina) raw data.

### 1. Reviewing Scientific Literature and Metadata
* **What is an Accession Number?** Unique identifier codes starting with SRR or ERR, shared in a study's methods or "Data Availability" section, that let us go directly to the raw sequencing data for that study in public databases.
* **Why Metadata Review Matters:** Searching this code in the database reveals critical information such as which platform the data came from (Illumina, Ion Torrent, or Nanopore), the reading strategy (Paired-end or Single-end), and file size. This information determines the core strategy of our analysis.

### 2. Sequencing Generations: Short Read and Long Read
We split sequencing technologies into two groups based on **read length**; this split determines which quality control tool we'll choose.
* **Short-read:** Covers the first and second generations; reads range from a few hundred bases up to ~1 kb.
  * *First Generation — Sanger (1977):* ~500-1000 bases, low throughput, high accuracy, the historical gold standard. The instrument outputs a chromatogram (.ab1/.scf) waveform; base-calling converts it to **FASTQ**.
  * *Second Generation — Illumina / Ion Torrent (NGS):* ~50-500 bases, very high throughput; today's gold standard. The instrument output is directly in **FASTQ** format.
  > A small detail: a single Sanger read (~1 kb) is actually *longer* than a single Illumina read (~150 bases); still, both are grouped in the same "short-read" class because they're both short compared to the third generation's "long reads" (tens of thousands of bases).
* **Long-read — Third Generation:** PacBio and Oxford Nanopore. Reads a single molecule in real time, tens of thousands of bases long.

Today the data we'll examine with **FastQC** is short-read (Illumina); the data we'll examine with **NanoPlot** is long-read (Nanopore). We'll come back to the Sanger origin of the "Phred quality score" concept at the end of the lesson, in the report evaluation section.

### Expert Note: Sequencing Technology Comparison Matrix
Knowing which technology produced the raw data directly affects the algorithm and parameters you'll choose when planning your analysis strategy. Thanks to current protocols, third-generation sequencing's historical "high error" problem has largely disappeared, reaching (and in some cases exceeding) short-read accuracy standards:

| Feature | 1st Gen (Sanger) | 2nd Gen (Illumina) | 3rd Gen (Oxford Nanopore / Modern PacBio) |
| :--- | :--- | :--- | :--- |
| **Read Length** | Medium (~500-1000 bases) | Short (~50-300 bases) | Very Long (10 kb - 1 Mb+) |
| **Throughput** | Very Low | Very High (Gigabytes/Terabytes) | High / Very High |
| **Raw Read Accuracy** | Very High (>99.9%) | Very High (>99.9%) | **High (>99.0% - 99.9%+)** |
| **Dominant Error Type** | Signal decay | Substitution | Deletion/insertion in homopolymer regions |
| **Typical Use** | Single gene / plasmid verification | Variant analysis, RNA-Seq, de novo genome | Large structural variants, difficult genome regions, methylation |

---

### Anatomy of the FASTQ Format
Before moving on to raw data quality control, let's remember that the **FASTQ** file we're about to X-ray is made up of a standard 4-line block for each read:

```text
Line 1: @SRR13680736.1 1/1 (Unique Read Identifier/Header - contains instrument and coordinate info)
Line 2: NTNGAATTTNGNTAAAAAATTTTTTTTTTTNGGGGGGGGGGGGGGGGGGGGGG (Nucleotide sequence)
Line 3: + (Separator line, sometimes repeats the header from line 1)
Line 4: F#FDFFAF#F#F#FFFFFFFFFFFFFFFF#FFFFFFFFFFFFFFFFFFFFFF (ASCII character values of the Phred quality scores)
```

### 3. Bringing the Data Into the Work Environment (Downloading Data)
Genomic data is very large. So instead of downloading data to our own personal computer, we'll download it directly to Google Colab's servers using Linux terminal commands.

In [ ]:
# COLAB & TERMINAL GUIDE: To send a command to the Linux terminal running behind Colab, we put an exclamation mark (!) at the start of the line.
# In a real Linux terminal, these commands would NOT have the exclamation mark.

# LINUX TERMINAL COMMAND: 'wget' (Web Get) is the basic tool for downloading a file from the internet directly to a server.

#    PURPOSE OF THIS CELL (DEMONSTRATION): This is how you would download the FULL raw dataset in a real project.
#    The links below are the full data files for our paper (Illumina ~441 MB, Nanopore ~2.1 GB).
#    Since they're large, we'll watch the download START and then STOP it with the ■ (Stop) button;
#    then in the next step we'll switch to a DOWNSAMPLING strategy for speed.

print("Downloading the FULL raw data... (large files - we'll STOP once we see it start)\n")

# Downloading the full data into a separate folder so it doesn't mix with the downsampled files:
# Short-read (Illumina, paired-end):
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_1.fastq.gz"
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_2.fastq.gz"

# Long-read (Nanopore, PacBio single-end):
!wget -P full_data_example "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR13680735/SRR13680735"

# Once the download starts progressing, stop the cell with the ■ (Stop) button above.
#    A half-downloaded file is harmless; in the next step we'll work with small downsampled files.

#### 3.0.a - Downloading the Full Short-Read (Illumina) Data [DEMONSTRATION]
You'll search for the short-read (Illumina) accession on SRA/ENA and copy the FASTQ download link yourself. We'll stop the download once it starts.

https://pmc.ncbi.nlm.nih.gov/articles/PMC8063646/

Because Illumina is Paired-End, it exists on the server as two separate files, _1 and _2:

R1 (Forward Read): http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_1.fastq.gz

R2 (Reverse Read): http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_2.fastq.gz


In [ ]:
# COLAB & TERMINAL GUIDE: To send a command to Colab's background terminal, we put an exclamation mark (!) at the start of the line.
# In a real Linux terminal, there would be NO exclamation mark at the start.

# GOAL (DEMONSTRATION): This is how you can download the ENTIRE SHORT-READ (Illumina) data in a real project.
#    Find the link YOURSELF: search for the Illumina accession on NCBI SRA / ENA; copy the FASTQ link
#    from the "Data access" or "FASTA/FASTQ download" tab.
#    Because Illumina is paired-end, there are TWO file links, _1 and _2.

print("Downloading the full SHORT-READ (Illumina) data... (large file - we'll STOP once we see it start)\n")

# APPLY: Remove the brackets and paste the FASTQ links you copied from SRA/ENA inside the quotes.
# (Put the full data in a separate folder so it doesn't mix with the downsampled files.)
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_1.fastq.gz"
!wget -P full_data_example "http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736_2.fastq.gz"

# Once the download starts progressing, stop it with ■ (Stop). A half-downloaded file is harmless.

#### 3.0.b - Downloading the Full Long-Read (Nanopore, PacBio) Data [DEMONSTRATION]
Using the same method, this time you'll search for the long-read (Oxford Nanopore) accession and copy the FASTQ link. Because Nanopore is single-end, there is only one file link.

Because Nanopore is Single-End, it exists on the server as a single file, without any direction suffix (like _1):

Single file: http://ftp.sra.ebi.ac.uk/vol1/fastq/SRR136/036/SRR13680736/SRR13680736.fastq.gz

In [ ]:
# COLAB & TERMINAL GUIDE: since this is a terminal command, we put an exclamation mark (!) at the start; a real terminal would NOT have it.

# GOAL (DEMONSTRATION): the method for downloading the ENTIRE LONG-READ (Nanopore) data.
#    Find the link YOURSELF: search for the Nanopore accession on NCBI SRA / ENA and copy the FASTQ link.
#    Since Nanopore is single-end there's a single read file (but the ENA server automatically appends _1 at the end).

print("Downloading the full LONG-READ (Nanopore, PacBio) data... (the largest file - we'll STOP once we see it start)\n")

# APPLY: paste the FASTQ link you copied from SRA/ENA inside the quotes.
!wget -P full_data_example "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR13680735/SRR13680735"

# Once the download starts progressing, stop it with ■ (Stop). A half-downloaded file is harmless.

### 3.1. Real-World Data and the Downsampling Strategy

When working on bioinformatics projects, the raw data we pull from public databases (NCBI SRA / ENA) is usually gigabytes in size. For example, the raw data sizes for the current hybrid sequencing paper we're using as our reference today (the *Gluconobacter cerinus* genome project) are as follows:
* **Short-read (Illumina NovaSeq 6000 - SRR13680736):** ~441 MB
* **Long-read (Oxford Nanopore MinION - SRR13680735):** ~2.1 GB

**Why Do We Downsample?**
In live analysis environments or training sessions, downloading these huge files in full just to test whether the software and quality control tools (FastQC, NanoPlot) work is a big waste of time and bandwidth.

Instead, pulling just a small slice (say, the first 50,000 reads) out of millions of reads directly from the database server is called **Downsampling**. Quality control tools looking at these 50,000 reads can give us a 99% accurate picture of the data's overall health.

> **Important Distinction:** This method is only used to save time during testing and quality control. If we were doing a real genome assembly or variant analysis, we would have to download the full data.

### The Statistical Justification for Downsampling: Why Are the First 50,000 Reads Enough?
On a sequencing instrument's flow cell, library fragments bind to the surface and cluster completely randomly (stochastically). So the first 50,000 reads of a FASTQ file are a homogeneous **sample** that reflects the technical and chemical course of that sequencing run. Under this reasoning, a systematic error in the pool (e.g. a clog on the flow cell, adapter contamination, or a base-calling cycle error) will show up statistically just as clearly within the first few thousand reads.

### 3.2. Installing the SRA Toolkit
To perform this downsampling, we'll install NCBI's official **SRA Toolkit** package on our cloud server.

In [ ]:
# COLAB & TERMINAL GUIDE: to send a package manager command to the Linux virtual machine running on Colab, we put an exclamation mark (!) at the start of the line.
# Later, when installing this tool on a real Linux terminal on your own computer, there will be NO EXCLAMATION MARK at the start (just 'sudo apt-get...').

# NCBI's official data management and download tool, sra-toolkit, is being installed on our server:
!sudo apt-get install sra-toolkit -y

### 3.3. Pulling a Downsample Directly With the SRA Toolkit

Installation is done. Now, using the SRA Toolkit component `fastq-dump`, we'll save only the first 50,000 reads to our server without downloading the full files. Here's what the parameters we'll use mean:

* `--split-files`: since Illumina data is Paired-End, this automatically separates the forward and reverse reads and adds `_1.fastq`, `_2.fastq` suffixes. This suffix does not get added for Single-End data like Nanopore.
  * **Bioinformatics tip:** if you forget this parameter, the forward and reverse reads get appended into a single FASTQ file (interleaved), and alignment (mapping) tools will error out when processing this data.
* `-X 50000`: tells the server "Don't send me the whole file, just give me the first 50,000 reads (spots) and close the connection." This way, only a few megabytes of test data download in seconds instead of gigabytes.

In [ ]:
# COLAB & TERMINAL GUIDE: since fastq-dump is a terminal tool, we put an exclamation mark (!) at the start of the Colab cell.
# When this command is run on a real Linux terminal screen later, it is written WITHOUT the exclamation mark.

print("Pulling 50,000-read DOWNSAMPLES from the NCBI SRA database for this paper...")
print("Since only a small slice is pulled instead of the full files, this will take seconds. Please wait...\n")

# 1. SHORT-READ (Illumina - Paired End) downsample:
!fastq-dump --split-files -X 50000 SRR13680736

# 2. LONG-READ (Nanopore - Single End) downsample:
!fastq-dump -X 50000 SRR13680735

print("\nDownsampled download completed successfully!")
print("Please click the folder icon in the left menu and check the actual names of the downloaded (.fastq) files yourself.")

# 4. Setting Up Short-Read Quality Control (FastQC)
We're adding FastQC, the industry-standard tool for quality control of short-read sequencing data, to our system.

> **Colab and Virtual Server Connection:** The Google Colab environment we're currently working in is a Linux virtual machine temporarily allocated to us by Google. The package manager command we're about to write will install the FastQC tool directly on this virtual machine.

In [ ]:
# COLAB & TERMINAL GUIDE: we use an exclamation mark (!) to call Linux's package manager (apt-get) through Colab.
# But when writing these commands on a real Linux terminal, there is NO exclamation mark (!) (just sudo apt-get...).

# TOOL PARAMETER: the '-y' (yes) parameter next to the 'install' command automatically confirms the installation.

!sudo apt-get update
!sudo apt-get install fastqc -y

### 4. Analyzing Raw Data With FastQC
Time to run the FastQC tool we successfully installed and examine the base-by-base quality distribution of our downloaded raw short-read data!

> **APPLICATION AND PEDAGOGICAL DISCOVERY NOTE:** In bioinformatics we never run code by rote. Now click the file explorer (folder) icon in the left panel. See the actual name and extension of the short-read file you downloaded with your own eyes (since this is a raw data file, it may be directly `.fastq` or, if compressed, `.fastq.gz`). In the code cell below, completely delete the auto-suggested bracketed placeholder and manually type the file name and extension yourself!

In bioinformatics, paired-end sequencing data always comes in a pair, and the naming convention is very strict:
* **File with `_1` (or R1) suffix:** **Forward** Read
* **File with `_2` (or R2) suffix:** **Reverse** Read

### Concept: What Is Paired-End Reading?
After DNA is broken into fragments (inserts), if the instrument starts from just one end of the fragment and reads a certain length (e.g. 150 bases), this is called **Single-End**. Reading in from both the left end (Forward - R1) and the right end (Reverse - R2) of the fragment is called **Paired-End**. Since the R2 read starts chemically later and the clusters on the flow cell degrade over time, it's completely normal biological/technical behavior to observe in a FastQC report that **R2's quality curve drops faster than R1's**.

> **APPLICATION NOTE:** In bioinformatics we never run code by rote. Now click the file explorer (folder) icon in the left panel. Find the short-read files you downloaded.
>
> In the code cell below, completely delete the bracketed placeholders; first manually copy and type the name of the Forward read file ending in **`_1`**, then leave **ONE SPACE** and type the name of the Reverse read file ending in **`_2`** from the left panel!

In [ ]:
# COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the folder-creation (mkdir) and analysis-triggering (fastqc) commands
# to run them in the Colab terminal. No exclamation mark is used in a real terminal.

# GUIDE: the 'mkdir' (Make Directory) command creates a folder. The '-p' (parents) parameter is a very critical
# safety measure in bioinformatics workflows; it stops the command from erroring out and halting the whole analysis if the folder already exists.
!mkdir -p qc_reports

# TOOL PARAMETERS AND LOGIC:
# 1. FastQC can process several files side by side at the same time (in parallel). Just leave a space between the file paths.
# 2. The '-o' (Output) parameter tells FastQC, "Which folder should I write the .html (visual report) and .zip (raw statistics) files to?"

# APPLY: manually type the actual names and extensions of the paired-end short-read files you saw in the
# left-panel file explorer below, first 1 (Forward) then 2 (Reverse), with ONE SPACE between them!

!fastqc /content/SRR13680736_1.fastq /content/SRR13680736_2.fastq -o qc_reports/

## LESSON 5: The World of Long Reads and Setting Up NanoPlot

### GOAL OF LESSON 5:
Understanding the unique nature of third-generation (long-read - Oxford Nanopore and PacBio) data, and producing a quality control report by installing the `NanoPlot` tool designed specifically for this data.

### 1. Why Do We Choose a Different Tool?
Third-generation data can reach tens of thousands of bases in length, by its very nature. But alongside this great length, raw error rates can be somewhat higher than short reads. Since FastQC is designed around short-read architecture, it can't fully represent the N50 value or the real-time length distribution of long reads. That's why we're installing `NanoPlot`, a tool designed specifically for the long-read world.

In [ ]:
# COLAB & TERMINAL GUIDE: NanoPlot is a Python library. We put an exclamation mark (!) at the start of the line to run
# the Python package manager 'pip' command via Colab. You'll write it without the exclamation mark on a real terminal.

print("Installing the NanoPlot tool and all required Python libraries...")
!pip install NanoPlot
print("\nGreat! Our NanoPlot installation completed successfully.")

### 2. Analyzing Raw Data With NanoPlot
Let's run the NanoPlot tool we just installed and get the statistical summary and quality map of our long-read data.

> **APPLICATION AND PEDAGOGICAL DISCOVERY NOTE:** Just like the previous step, open the file explorer in the left panel. Check the actual name and extension of the long-read file you downloaded with your own hand (depending on the instrument output, it could be `.fastq` or a compressed `.fastq.gz` archive). In the code cell below, completely delete the bracketed target field and manually type the file name and extension!

In [ ]:
# COLAB & TERMINAL GUIDE: we use an exclamation mark (!) to start the NanoPlot analysis via Colab. No exclamation mark on a real terminal.

# TOOL PARAMETERS AND LOGIC:
# 1. The '--fastq' parameter tells the tool the format of the file it will analyze.
# 2. The '-o' (Output) parameter sets the destination folder for the results.
# 3. The '--threads 2' parameter speeds up the process by telling the tool "use 2 processor cores of the computer at the same time for this analysis."

# APPLY: manually type the actual name and extension (.fastq or .fastq.gz) of the long-read (Nanopore) file
# you saw in the left-panel file explorer below!

!NanoPlot --fastq /content/SRR13680735.fastq -o qc_reports/nanoplot_result --threads 2

## LESSON 6: Multi-Tool Reporting and Overall Evaluation

### GOAL OF LESSON 6:
Combining the scattered quality control outputs we got from different tools (FastQC, NanoPlot, etc.) into a single, interactive, professional HTML report using `MultiQC`, and analyzing the state of our raw data.

### 1. Combining Reports With MultiQC
Right now we have FastQC reports for the short reads and NanoPlot reports for the long reads. In a real project, examining dozens of different samples one by one can be time-consuming. This is exactly where `MultiQC` comes to the rescue. MultiQC automatically scans all the different analysis outputs in our workspace and combines them into a single interactive report.

In [ ]:
# COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the line to install the MultiQC tool
# through the Colab terminal interface. You'll write it without the exclamation mark on a real terminal.

print("Installing the MultiQC tool...")
!pip install multiqc
print("\nMultiQC installation completed successfully!")

Now let's run MultiQC to combine the reports.

> **A Small Linux Note:** In the Linux file system, a single dot (`.`) means "the folder I'm currently in." Two dots side by side (`..`) represent the parent folder. This is one of the most useful bits of practical knowledge you'll reach for when specifying paths in the terminal.

In [ ]:
# COLAB & TERMINAL GUIDE: we put an exclamation mark (!) at the start of the cell code to trigger MultiQC
# and have it scan the folder. No exclamation mark is used on a real terminal.

# TOOL PARAMETERS AND LOGIC:
# 1. The first 'qc_reports/' in the command tells MultiQC, "Go into this folder and find all the analysis logs you recognize (FastQC, NanoPlot, etc.)"
# 2. The 'qc_reports/' after the '-o' (Output) parameter means "Combine all this data and save the final HTML report you'll produce into this same folder too."
# Bioinformatics note: MultiQC is a very 'smart' tool. Even if you give it a folder full of thousands of unrelated files, it will only find and pick out the outputs of the bioinformatics tools it supports.

print("MultiQC is scanning all subfolders and combining the reports...")
!multiqc qc_reports/ -o qc_reports/

print("\nSuccess! Our combined interactive quality control report is ready.")
print("Please open the 'qc_reports' folder in the left panel, right-click the newly created 'multiqc_report.html' file and download it to your computer.")

## Evaluating the Reports and an Analytical Perspective

Dear researchers, please go to the `qc_reports` folder in the file explorer on the left. After right-clicking `multiqc_report.html` and downloading it to your computer, open it in your browser.

### What Do We Look At Through a Bioinformatician's Eyes?
1. **Sequence Quality Histograms (Short Read):** Are the Phred quality scores of the bases in the short-read file you typed in in the green zone (Q30 and above)? Where do the drooping, quality-dropping regions begin?

2. **The Secret in the Encoding Field:** In the **Encoding** row of the "Basic Statistics" table, you'll usually see **"Sanger / Illumina 1.9"**. Let's clear up the most critical misunderstanding up front: this row does **not** tell you which *instrument* the data came from. It shows the **encoding scheme** the quality scores are stored in within the file — namely **Phred+33**. This scheme's name comes from the fact that the Phred quality system was historically first developed in Sanger projects. Today, Illumina, Nanopore, and PacBio all inherit this standard; so **a Nanopore FASTQ file** will also show the same "Sanger / Illumina 1.9" text. In short: encoding = the quality encoding language, not the platform.

3. **Is Our Data Ready for QC? (Bioinformatics Checklist):**
   * **Extension Check:** If the file is `.fastq` or `.fastq.gz`, it has been base-called and contains quality scores, so it's raw data and ready for FastQC. FastQC reads `.fastq.gz` files directly without needing to extract the archive. If the file is `.fasta`, it has no quality scores and isn't suitable for QC. Instrument-specific raw signal formats (e.g. Sanger's `.ab1`/`.scf` chromatogram or Nanopore's `.fast5`/`.pod5` files) must first be converted to FASTQ via base-calling.
   * **Database Metadata:** When you search the accession on SRA/ENA, seeing the instrument you expect (e.g. `ILLUMINA` or `OXFORD_NANOPORE`) in the "Platform" field, and the data being offered as `FASTQ`, confirms it fits our workflow.
   * **A First Look With the Linux Terminal:** After downloading the data to the server, you can manually check the first 4 lines by looking at the name in the left panel and typing `!head -n 4 file_name.fastq` (or, if compressed, `!gzip -cd file_name.fastq.gz | head -n 4`). (We put a `!` at the start of the command because we're in Colab; on a real Linux terminal you'd write `head -n 4 file_name.fastq` without the `!`.) If a header starting with `@`, a nucleotide sequence, a `+` sign and quality characters come in that order, the file has been successfully converted to FASTQ format.

4. **Read Length & N50 (Long Read):** What's the longest read in the long-read file you typed in? Does the read length threshold that makes up half our data (N50) fit our biological hypothesis?

5. **Adapter Content:** Have technical adapter sequences read during sequencing been left inside the raw data?

### Preparing for the Next Lesson
We've essentially taken an X-ray of our raw data and identified on the report where things are "dirty" or "low quality." In our next lesson, we'll see why we'd get errors if we sent this data straight into alignment (mapping) as-is, and we'll learn to clean up this noise (Trimming & Filtering) with the tools in our **BIF201 toolkit (Tool List)**:

* **For Short Reads:** we'll cut technical adapters with `Cutadapt` and run our data through a quality filter. *(Extra info: in the literature and in modern workflows you'll also often come across `fastp`, a fast, all-in-one tool for this job, but in our training we'll use Cutadapt, the gold standard that best teaches the parameter logic.)*
* **For Long Reads:** we'll shave off Nanopore adapters with `Porechop`, and select the longest and highest-quality reads with `Filtlong`.

> **BIF201 Workflow:** Once we have our clean reads, we'll get to know the industry-standard algorithms for aligning these reads to a reference genome: **BWA-MEM** for short reads, and **Minimap2** for long reads!

---
### A Trip Into the Real Linux World
If you'd like to run these commands later on your own computer or on a lab server:
* **Windows Users:** can download **WSL (Windows Subsystem for Linux)** for free from the Microsoft Store and install Ubuntu inside it.
* **Mac Users:** can open the **Terminal** app already built into their system and run these same commands directly (using `brew` instead of `apt`).

---

### References

* **FastQC:** Andrews S. (2010). FastQC: A quality control tool for high throughput sequence data.
  * **Link:** [http://www.bioinformatics.babraham.ac.uk/projects/fastqc/](http://www.bioinformatics.babraham.ac.uk/projects/fastqc/)

* **NanoPlot:** De Coster W. et al. (2018). NanoPack: visualizing and processing long-read sequencing data. Bioinformatics, 34(15), 2666-2669.
  * **DOI:** [https://doi.org/10.1093/bioinformatics/bty149](https://doi.org/10.1093/bioinformatics/bty149)

* **MultiQC:** Ewels P. et al. (2016). MultiQC: summarize analysis results for multiple tools and samples in a single report. Bioinformatics, 32(19), 3047-3048.
  * **DOI:** [https://doi.org/10.1093/bioinformatics/btw354](https://doi.org/10.1093/bioinformatics/btw354)

* **Cutadapt:** Martin, M. (2011). Cutadapt removes adapter sequences from high-throughput sequencing reads. EMBnet.journal, 17(1), 10-12. *(Industry standard for short-read adapter trimming.)*
  * **DOI:** [https://doi.org/10.14806/ej.17.1.200](https://doi.org/10.14806/ej.17.1.200)

* **Porechop:** Wick, R. R. (2017). Porechop: adapter depleter for Oxford Nanopore reads. GitHub Repository. *(Gold-standard adapter detection and trimming tool developed for Nanopore data.)*
  * **Link:** [https://github.com/rrwick/Porechop](https://github.com/rrwick/Porechop)

* **Filtlong:** Wick, R. R. (2017). Filtlong: quality filtering tool for long reads. GitHub Repository. *(Core tool for quality- and length-based filtering of long reads.)*
  * **Link:** [https://github.com/rrwick/Filtlong](https://github.com/rrwick/Filtlong)